In [3]:
pip install git+https://github.com/damonge/schnell.git

  Cloning https://github.com/damonge/schnell.git to c:\users\tiemo\appdata\local\temp\pip-req-build-6z8775vy
Note: you may need to restart the kernel to use updated packages.


  ERROR: Error [WinError 2] Het systeem kan het opgegeven bestand niet vinden while executing command git version
ERROR: Cannot find command 'git' - do you have 'git' installed and in your PATH?


In [2]:
# Install dependencies if needed (run once):
# pip install astropy astropy-healpix scipy

#imports

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import sph_harm
import astropy_healpix as ahp
import astropy.units as u

from schnell.mapping import MapCalculator
from schnell.detector import LISADetector

ModuleNotFoundError: No module named 'schnell'

In [ ]:
#constanten enzo

C_LIGHT = 299_792_458.0
L_ARM   = 2.5e9
F_STAR  = C_LIGHT / (2.0 * np.pi * L_ARM)

MPC_M     = 3.0856775814913673e22
H0_OVER_h = (100.0 * 1000.0) / MPC_M

P_OMS_PM = 15.0
A_ACC_FM = 3.0

In [ ]:
# ── HEALPix + SHT helper replacements (no healpy, no ducc0) ──────────────────
#
# Pixel ops:  astropy_healpix  (pure Python, no DLL)
# SHT:        scipy.special.sph_harm  (pure Python)
#
# alm packing (identical to healpy):
#   index = m*(2*lmax+1-m)//2 + ell   for  0 <= m <= ell <= lmax
#
# The SHT is a direct pixelised quadrature:
#   a_lm = (4pi/Npix) * sum_p  f(p) * conj(Y_lm(theta_p, phi_p))
# Iterative refinement corrects for pixel-window bias.
# The Y_lm matrix is cached after the first build.

def nside2npix(nside):
    return ahp.nside_to_npix(nside)

def pix2ang(nside, ipix):
    """Return (theta, phi) in radians – RING scheme, same as healpy."""
    ipix = np.asarray(ipix)
    lon, lat = ahp.healpix_to_lonlat(ipix, nside, order='ring')
    theta = (90.0 * u.deg - lat).to(u.rad).value   # colatitude 0..pi
    phi   = lon.to(u.rad).value                     # longitude  0..2pi
    return theta, phi

def alm_getidx(lmax, ell, m):
    """Packed alm index (healpy convention, m <= ell)."""
    return m * (2 * lmax + 1 - m) // 2 + ell

# Cache: key=(nside,lmax) -> Y matrix of shape (nalm, npix)
_ylm_cache = {}

def _get_ylm_matrix(nside, lmax, theta, phi):
    """
    Build (and cache) the Y_lm matrix of shape (nalm, npix).
    Y[idx, p] = Y_{ell,m}(theta_p, phi_p)  for packed index idx.
    Built once per (nside, lmax); subsequent calls return instantly.
    """
    key = (nside, lmax)
    if key in _ylm_cache:
        return _ylm_cache[key]
    nalm = (lmax + 1) * (lmax + 2) // 2
    npix = theta.size
    Y = np.zeros((nalm, npix), dtype=np.complex128)
    for ell in range(lmax + 1):
        for m in range(0, ell + 1):
            idx = alm_getidx(lmax, ell, m)
            # scipy: sph_harm(m, ell, phi, theta)  (azimuth first, then polar)
            Y[idx] = sph_harm(m, ell, phi, theta)
    _ylm_cache[key] = Y
    print(f"[SHT] Y_lm matrix built: nside={nside}, lmax={lmax}, shape={Y.shape}")
    return Y

def map2alm_real(hp_map, lmax, niter=3, _theta=None, _phi=None):
    """
    Forward SHT: real HEALPix RING map -> packed complex alm (healpy convention).
    Uses equal-area pixel quadrature with iterative refinement.
    """
    hp_map = np.asarray(hp_map, dtype=np.float64)
    npix   = hp_map.size
    nside  = ahp.npix_to_nside(npix)
    if _theta is None or _phi is None:
        _theta, _phi = pix2ang(nside, np.arange(npix))
    Y = _get_ylm_matrix(nside, lmax, _theta, _phi)   # (nalm, npix)
    w = 4.0 * np.pi / npix                            # pixel solid angle
    alm = (Y.conj() @ hp_map) * w
    for _ in range(niter - 1):
        residual = hp_map - _alm2map_internal(alm, Y, lmax)
        alm += (Y.conj() @ residual) * w
    return alm

def _alm2map_internal(alm, Y, lmax):
    """Synthesis using pre-built Y matrix. Returns real-valued map."""
    alm = np.asarray(alm, dtype=np.complex128)
    hp_map = Y.T @ alm
    # Add negative-m terms (for real input maps: a_{l,-m} = (-1)^m conj(a_{lm}))
    for ell in range(1, lmax + 1):
        for m in range(1, ell + 1):
            idx   = alm_getidx(lmax, ell, m)
            a_neg = ((-1)**m) * np.conj(alm[idx])
            hp_map += a_neg * np.conj(Y[idx])
    return np.real(hp_map)

def alm2map_real(alm, nside, lmax, _theta=None, _phi=None):
    """Inverse SHT: packed complex alm -> real HEALPix RING map."""
    if _theta is None or _phi is None:
        _theta, _phi = pix2ang(nside, np.arange(nside2npix(nside)))
    Y = _get_ylm_matrix(nside, lmax, _theta, _phi)
    return _alm2map_internal(alm, Y, lmax)

def map2alm_complex(hp_map, lmax, iter=3):
    """SHT for a complex-valued sky map: split into Re and Im parts."""
    hp_map = np.asarray(hp_map)
    npix   = hp_map.size
    nside  = ahp.npix_to_nside(npix)
    theta, phi = pix2ang(nside, np.arange(npix))
    alm_re = map2alm_real(np.real(hp_map), lmax=lmax, niter=iter,
                          _theta=theta, _phi=phi)
    alm_im = map2alm_real(np.imag(hp_map), lmax=lmax, niter=iter,
                          _theta=theta, _phi=phi)
    return alm_re + 1j * alm_im

def alm2map_complex(alm, nside, lmax):
    """Inverse SHT for complex alm coefficients."""
    alm   = np.ascontiguousarray(alm.astype(np.complex128, copy=False))
    theta, phi = pix2ang(nside, np.arange(nside2npix(nside)))
    Y = _get_ylm_matrix(nside, lmax, theta, phi)
    # Synthesise real and imaginary parts separately
    m_re = _alm2map_internal(np.real(alm).astype(complex), Y, lmax)
    m_im = _alm2map_internal((1j * np.imag(alm)), Y, lmax)
    return m_re + 1j * m_im

def isolate_ell_from_map(hp_map, ell, lmax, nside, iter=3):
    alm = map2alm_complex(hp_map, lmax=lmax, iter=iter).astype(np.complex128)
    alm_filt = np.zeros_like(alm)
    for m in range(0, ell + 1):
        idx = alm_getidx(lmax, ell, m)
        alm_filt[idx] = alm[idx]
    return alm2map_complex(alm_filt, nside=nside, lmax=lmax)

def lowpass_ell_from_map(hp_map, ell_max, lmax, nside, iter=3):
    alm  = map2alm_complex(hp_map, lmax=lmax, iter=iter)
    alm2 = alm.copy()
    for ell in range(ell_max + 1, lmax + 1):
        for m in range(0, ell + 1):
            alm2[alm_getidx(lmax, ell, m)] = 0.0
    return alm2map_complex(alm2, nside=nside, lmax=lmax)

In [ ]:
#funties vanuit appendix B; N_TT tilde en N_AE tilde

def S_oms(f):
    f = np.asarray(f, dtype=float)
    return (P_OMS_PM * 1e-12)**2 * (1.0 + (2.0e-3 / f)**4)

def S_acc(f):
    f = np.asarray(f, dtype=float)
    return (A_ACC_FM * 1e-15)**2 * (1.0 + (0.4e-3 / f)**2) * (1.0 + (f / 8.0e-3)**4)

def N_tilde_AE(f):
    f = np.asarray(f, dtype=float)
    x = f / F_STAR
    cosx = np.cos(x)
    term_oms = 0.5 * (2.0 + cosx) * (S_oms(f) / (L_ARM**2))
    term_acc = 2.0 * (1.0 + cosx + cosx**2) * (S_acc(f) / (L_ARM**2)) * (1.0 / (2.0*np.pi*f))**4
    return term_oms + term_acc

def N_tilde_T(f):
    f = np.asarray(f, dtype=float)
    x = f / F_STAR
    cosx = np.cos(x)
    term_oms = (1.0 - cosx) * (S_oms(f) / (L_ARM**2))
    term_acc = 2.0 * (1.0 - cosx)**2 * (S_acc(f) / (L_ARM**2)) * (1.0 / (2.0*np.pi*f))**4
    return term_oms + term_acc

In [ ]:
C_AET = np.array([
    [-1/np.sqrt(2),  0.0,          1/np.sqrt(2)],   # A
    [ 1/np.sqrt(6), -2/np.sqrt(6), 1/np.sqrt(6)],   # E
    [ 1/np.sqrt(3),  1/np.sqrt(3), 1/np.sqrt(3)],   # T
    ], dtype=float)

OIDX = {"A": 0, "E": 1, "T": 2}


def Rlm_OOp_from_rlmij(rlm_ij_mge0, ell, m, O, Op):
    cO  = C_AET[OIDX[O]]
    cOp = C_AET[OIDX[Op]]
    return np.sum(cO[:, None] * cOp[None, :] * rlm_ij_mge0)  # Eq. 4.27


def Rell_OOp_from_rlmij_list(rlm_ij_list_mge0, ell, O, Op):  # Eq. 4.28
    z0 = Rlm_OOp_from_rlmij(rlm_ij_list_mge0[0], ell, 0, O, Op)
    s  = np.abs(z0)**2
    for m in range(1, ell + 1):
        zm = Rlm_OOp_from_rlmij(rlm_ij_list_mge0[m], ell, m, O, Op)
        s += 2.0 * np.abs(zm)**2
    return np.sqrt(s)

In [ ]:
#calibratie functie

def calibrate_scale_from_R0AA(mc, theta, phi, lmax, f_cal=1e-5, t0=0.0, target=9.0/20.0):
    npix = theta.size

    ant_ij = [[None]*3 for _ in range(3)]
    for i in range(3):
        for j in range(3):
            ant = mc.get_antenna(i, j, t0, float(f_cal), theta, phi, pol=False, inc_baseline=True)
            ant = np.asarray(ant).squeeze()
            if ant.shape != (npix,):
                raise RuntimeError(f"get_antenna returned {ant.shape}, expected {(npix,)}")
            ant_ij[i][j] = ant

    alm_ij = [[None]*3 for _ in range(3)]
    for i in range(3):
        for j in range(3):
            alm = map2alm_complex(ant_ij[i][j], lmax=lmax, iter=3)
            alm_ij[i][j] = alm / (8.0 * np.pi)

    idx   = alm_getidx(lmax, 0, 0)
    mat00 = np.zeros((3,3), dtype=np.complex128)
    for i in range(3):
        for j in range(3):
            mat00[i,j] = alm_ij[i][j][idx]

    R0AA = float(np.real(Rell_OOp_from_rlmij_list([mat00], ell=0, O="A", Op="A")))
    if R0AA <= 0:
        raise RuntimeError(f"Calibration failed: got R0AA={R0AA}")

    scale = target / R0AA
    print(f"[calibration] f_cal={f_cal:g} Hz  R0AA={R0AA:.6g}  target={target:.6g}  scale={scale:.6g}")
    return scale

In [ ]:
def compute_Rlm_ij(f_grid, lmax=10, nside=128, t0=0.0, calibrate=True, f_cal=1e-5):
    f_grid = np.asarray(f_grid, dtype=float)

    dets = [LISADetector(detector_id=i) for i in range(3)]
    mc   = MapCalculator(dets)

    npix       = nside2npix(nside)
    theta, phi = pix2ang(nside, np.arange(npix))

    # Build Y_lm matrix once up front (cached)
    _get_ylm_matrix(nside, lmax, theta, phi)

    scale = 1.0
    if calibrate:
        scale = calibrate_scale_from_R0AA(mc, theta, phi, lmax=lmax,
                                          f_cal=f_cal, t0=t0, target=9.0/20.0)

    def antenna_alm(i, j, f):
        ant = mc.get_antenna(i, j, t0, float(f), theta, phi, pol=False, inc_baseline=True)
        ant = np.asarray(ant).squeeze()
        if ant.shape != (npix,):
            raise RuntimeError(f"get_antenna returned {ant.shape} (expected {(npix,)})")
        alm = map2alm_complex(scale * ant, lmax=lmax, iter=3)
        return alm / (8.0 * np.pi)

    rlm_all = []
    for k, f in enumerate(f_grid):
        print(f"  freq {k+1}/{len(f_grid)}  f={f:.4g} Hz", end="\r")
        M = [[antenna_alm(i, j, f) for j in range(3)] for i in range(3)]
        rlm_all.append(M)
    print()
    return rlm_all

In [ ]:
def simple_mollview(map_vals, title="", nside=128):
    """Minimal Mollweide sky plot (replaces hp.mollview)."""
    t, p = pix2ang(nside, np.arange(nside2npix(nside)))
    lon  = np.degrees(p); lon[lon > 180] -= 360.0
    lat  = 90.0 - np.degrees(t)
    fig, ax = plt.subplots(1, 1, figsize=(8, 4),
                           subplot_kw={'projection': 'mollweide'})
    sc = ax.scatter(np.radians(lon), np.radians(lat),
                    c=np.real(map_vals), s=0.5, cmap='RdBu_r')
    plt.colorbar(sc, ax=ax, orientation='horizontal', pad=0.05)
    ax.set_title(title)
    ax.grid(True)
    plt.tight_layout()
    plt.show()

# Build one antenna sky map (baseline 0,1) and decompose by multipole
nside = 64   # lower nside keeps runtime manageable
lmax  = 10
t0    = 0.0
f     = 0.1

dets = [LISADetector(detector_id=i) for i in range(3)]
mc   = MapCalculator(dets)

npix       = nside2npix(nside)
theta, phi = pix2ang(nside, np.arange(npix))

ant = mc.get_antenna(0, 1, t0, float(f), theta, phi, pol=False, inc_baseline=True)
ant = np.asarray(ant).squeeze()

for ell in range(0, lmax + 1):
    m_ell = isolate_ell_from_map(ant, ell=ell, lmax=lmax, nside=nside, iter=3)
    simple_mollview(np.real(m_ell),
                    title=f"Re[A_01] multipole \u2113={ell}  f={f:g} Hz",
                    nside=nside)

In [ ]:
def omega_from_pair_threshold(f_grid, Rv, N1, N2, rel_thr=1e-18, abs_thr=0.0):
    Rv   = np.asarray(Rv, dtype=float)
    N1   = np.asarray(N1, dtype=float)
    N2   = np.asarray(N2, dtype=float)
    Rmax = np.max(Rv)
    thr  = max(abs_thr, rel_thr * Rmax) if Rmax > 0 else np.inf
    om   = np.full_like(f_grid, np.inf, dtype=float)
    mask = Rv > thr
    if np.any(mask):
        om[mask] = omega_channel_channel_tilde(
            f_grid[mask], Rv[mask], N1[mask], N2[mask]
        )
    return om

# functies 4.42 en 4.43
def omega_channel_channel_tilde(f, Rtilde_ell, Ntilde1, Ntilde2):
    pref = (4.0 * np.pi**2 * np.sqrt(4.0 * np.pi)) / (3.0 * (H0_OVER_h**2))
    return pref * (f**3) * (np.sqrt(Ntilde1 * Ntilde2) / Rtilde_ell)

def omega_combined_from_channels(omega_auto, omega_cross):
    inv2 = np.zeros_like(omega_auto[0], dtype=float)
    for om in omega_auto + omega_cross:
        inv2 += 1.0 / (om**2)
    return inv2**(-0.5)

In [ ]:
def recreate_figure9(f_grid, lmax, nside):
    f_grid = np.asarray(f_grid, dtype=float)

    rlm_all = compute_Rlm_ij(f_grid, lmax=lmax, nside=nside, t0=0.0,
                              calibrate=True, f_cal=1e-5)

    N_A   = N_tilde_AE(f_grid)
    N_E   = N_A.copy()
    N_T   = N_tilde_T(f_grid)
    N_map = {"A": N_A, "E": N_E, "T": N_T}
    Y00   = 1.0 / np.sqrt(4.0 * np.pi)

    pairs = [
        ("A", "A", "AA"), ("E", "E", "EE"), ("T", "T", "TT"),
        ("A", "E", "AE"), ("A", "T", "AT"), ("E", "T", "ET"),
    ]

    omega_ell = {}
    for ell in range(lmax + 1):
        print(f"computing ell={ell}...")
        Rell = {key: np.zeros_like(f_grid, dtype=float) for _, _, key in pairs}

        for i_f in range(len(f_grid)):
            rlm_ij_list_mge0 = []
            for m in range(0, ell + 1):
                mat = np.zeros((3, 3), dtype=np.complex128)
                idx = alm_getidx(lmax, ell, m)
                for i in range(3):
                    for j in range(3):
                        mat[i, j] = rlm_all[i_f][i][j][idx]
                if (ell + m) % 2 == 1:   # selection rule Eq. 4.21
                    mat[:, :] = 0.0
                rlm_ij_list_mge0.append(mat)

            for O, Op, key in pairs:
                Rell[key][i_f] = float(
                    np.real(Rell_OOp_from_rlmij_list(rlm_ij_list_mge0, ell, O, Op))
                )

        omega_auto  = [omega_from_pair_threshold(f_grid, Rell[O+O],
                           N_map[O], N_map[O], rel_thr=1e-20) for O in "AET"]
        omega_cross = [omega_from_pair_threshold(f_grid, Rell[key],
                           N_map[O1], N_map[O2], rel_thr=1e-20)
                       for O1, O2, key in [("A","E","AE"),("A","T","AT"),("E","T","ET")]]

        omega_ell[ell] = omega_combined_from_channels(omega_auto, omega_cross) * Y00

    return omega_ell

In [ ]:
#plotter broertje

f = np.logspace(-4, -0.25, 80)
curves = recreate_figure9(f, lmax=10, nside=64)

plt.figure(figsize=(8, 8))
for ell in range(0, 11):
    plt.loglog(f, curves[ell], label=fr"$\ell={ell}$")

plt.xlabel("Frecuency [Hz]")
plt.ylim([1e-13, 1e0])
plt.ylabel(r"$\Omega^{\ell}_{{\rm GW},n}(f)\,h^2 / \sqrt{4\pi}$")
plt.title("Recreation of Figure 9")
plt.grid(True, which="both", ls=":")
plt.legend(ncol=2, fontsize=10)
plt.tight_layout()
plt.show()